# Commodity Price Prediction — Full Pipeline v3

**Flow:** News (BT 5k) → FinBERT sentiment → keyword topic relevance →
event classification → polarity-corrected aggregation →
XGBoost / LightGBM / BinaryLSTM → evaluation

**Commodities:** `brent` · `wti` · `natural_gas` · `silver`
**Target:** next-day direction (up = 1 / down = 0)

**v3 changelog vs v2**
- Colab-ready: Google Drive paths, `pip install` guard cell
- `to_numeric` on price at load time (prevents str→log crash)
- Cell 5: direct column assignment (no many-to-many merge RAM crash)
- Cell 9: XGBoost + LightGBM with early stopping, graceful fallback
- Cell 11: `baseline_offset` slice so LSTM and baselines compare on same rows
- Cell 13: all models in ROC / accuracy plots

In [ ]:
# ── 0. Colab Setup (skip if running locally) ────────────────────────────────
import sys, subprocess

def _pip(*pkgs):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *pkgs])

try:
    import google.colab          # noqa: F401  — only runs inside Colab
    _pip('transformers', 'torch', 'scikit-learn',
         'xgboost', 'lightgbm', 'pandas', 'numpy', 'matplotlib')
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ModuleNotFoundError:
    IN_COLAB = False

print(f'Colab: {IN_COLAB}')

In [ ]:
# ── 1. Imports & Configuration ───────────────────────────────────────────────
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

warnings.filterwarnings('ignore')

# ── Paths ─────────────────────────────────────────────────────────────────────
# Colab  →  set DRIVE_PROJECT to your Drive folder
# Local  →  BASE is resolved automatically from the notebook location
try:
    _in_colab = 'google.colab' in str(get_ipython())   # type: ignore
except NameError:
    _in_colab = False

if _in_colab:
    BASE = Path('/content/drive/MyDrive/Project')   # ← change if needed
else:
    try:
        BASE = Path(__file__).resolve().parents[1]
    except NameError:
        BASE = Path.cwd().parent                    # Jupyter: nb in prediction/

DATA_DIR   = BASE / 'data' / 'raw'
PRICE_DIR  = DATA_DIR / 'daily_last_2y' / 'csv'
NEWS_FILE  = DATA_DIR / 'news' / '2y_daily' / 'bt_energy_commodities_2y_5k.jsonl'

OUT_DIR = BASE / 'prediction' / 'pipeline_outputs'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Hyper-parameters ──────────────────────────────────────────────────────────
TARGET_COMMODITY = 'brent'     # brent | wti | natural_gas | silver
SEQUENCE_LEN     = 10          # LSTM lookback (10 gives 3× more sequences than 30)
BATCH_SIZE       = 32
EPOCHS           = 100
LR               = 1e-3
TRAIN_RATIO      = 0.70
VAL_RATIO        = 0.15
EARLY_STOP_PAT   = 20

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {DEVICE}')
print(f'Target : {TARGET_COMMODITY}')
print(f'Base   : {BASE}')
print(f'Output : {OUT_DIR}')

In [ ]:
# ── 2. Load News + Trading-Day Assignment ────────────────────────────────────
from datetime import timedelta

CUTOFF_HOUR_UTC = 20   # ICE Brent settles ~20:30 UTC

def next_business_day(d):
    while d.weekday() >= 5:
        d += timedelta(days=1)
    return d

def assign_trading_day(dt_utc):
    d = dt_utc.date()
    if dt_utc.hour >= CUTOFF_HOUR_UTC:
        d += timedelta(days=1)
    return next_business_day(d)

def load_news(path: Path) -> pd.DataFrame:
    records = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    df = pd.DataFrame(records)
    df['published_utc'] = pd.to_datetime(df['published_at'], utc=True).dt.tz_convert(None)
    df['trading_day']   = df['published_utc'].apply(assign_trading_day)
    df['trading_day']   = pd.to_datetime(df['trading_day'])
    df['text'] = (df['title'].fillna('') + '. ' + df['summary'].fillna('')).str.strip('. ')
    return df[['trading_day', 'published_utc', 'text', 'title', 'url']].sort_values('trading_day').reset_index(drop=True)

news_df = load_news(NEWS_FILE)
print(f'News records : {len(news_df):,}')
print(f'Date range   : {news_df["trading_day"].min().date()} → {news_df["trading_day"].max().date()}')
shifted = (news_df['published_utc'].dt.hour >= CUTOFF_HOUR_UTC).sum()
print(f'Shifted past cutoff : {shifted:,}')
news_df.head(3)

In [ ]:
# ── 3. Load Price Data ───────────────────────────────────────────────────────
# CSVs have columns 'date' and 'value'. Rename + coerce to float immediately.

def load_prices(price_dir: Path, commodity: str) -> pd.DataFrame:
    files = sorted(price_dir.glob(f'{commodity}_daily_*.csv'))
    if not files:
        raise FileNotFoundError(f'No price CSV for {commodity} in {price_dir}')
    df = pd.read_csv(files[-1])
    df = df.rename(columns={'value': 'price'})
    df['price'] = pd.to_numeric(df['price'], errors='coerce')   # str → float guard
    df['date']  = pd.to_datetime(df['date'])
    df = df.sort_values('date').reset_index(drop=True)
    print(f'  {commodity:12s}: {len(df)} rows  '
          f'{df["date"].min().date()} → {df["date"].max().date()}')
    return df

print('Loading price data...')
price_df = load_prices(PRICE_DIR, TARGET_COMMODITY)
price_df.tail(3)

In [ ]:
# ── 4. FinBERT Sentiment Scoring ─────────────────────────────────────────────
SENTIMENT_CACHE = OUT_DIR / 'sentiment_cache.csv'

# Invalidate cache if saved with old 'date' column
if SENTIMENT_CACHE.exists():
    _cols = pd.read_csv(SENTIMENT_CACHE, nrows=0).columns.tolist()
    if 'trading_day' not in _cols:
        print('Old sentiment cache detected — deleting...')
        SENTIMENT_CACHE.unlink()

def score_finbert(texts, tokenizer, model, batch_size=64, device='cpu'):
    """Score = (P_pos - P_neg) × (1 - P_neu)  ∈ [-1, +1]
    ProsusAI/finbert label order: 0=positive, 1=negative, 2=neutral
    """
    model.eval()
    scores = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        enc = tokenizer(batch, padding=True, truncation=True,
                        max_length=128, return_tensors='pt')
        enc = {k: v.to(device) for k, v in enc.items()}
        with torch.no_grad():
            probs = torch.softmax(model(**enc).logits, dim=1).cpu().numpy()
        pos, neg, neu = probs[:, 0], probs[:, 1], probs[:, 2]
        scores.extend(((pos - neg) * (1 - neu)).tolist())
        if (i // batch_size) % 20 == 0:
            print(f'  FinBERT: {min(i+batch_size, len(texts))}/{len(texts)}', end='\r')
    print()
    return np.array(scores)

if SENTIMENT_CACHE.exists():
    print('Loading cached sentiment scores...')
    sent_df = pd.read_csv(SENTIMENT_CACHE, parse_dates=['trading_day'])
else:
    print('Running FinBERT (slow on CPU — cached after first run)...')
    tokenizer = AutoTokenizer.from_pretrained('ProsusAI/finbert')
    finbert   = AutoModelForSequenceClassification.from_pretrained('ProsusAI/finbert').to(DEVICE)
    news_df['sentiment_score'] = score_finbert(
        news_df['text'].tolist(), tokenizer, finbert, batch_size=64, device=DEVICE)
    sent_df = news_df[['trading_day', 'sentiment_score']].copy()
    sent_df.to_csv(SENTIMENT_CACHE, index=False)
    print(f'Saved {len(sent_df):,} scores → {SENTIMENT_CACHE.name}')

print(sent_df['sentiment_score'].describe().round(4))

In [ ]:
# ── 5. Topic Relevance + Event Classification ────────────────────────────────
#
# Stage 1: keyword-based relevance score (replaces sentence-transformer)
#   BT feed is already scoped to Energy & Commodities so neural embeddings
#   added ~90 min CPU time with near-zero variance (all scores 0.7-0.9).
#   Keyword hit-count runs in < 1 second.
#
# Stage 2: 8-category event taxonomy (multi-label keyword matching)

TOPIC_MAP = {
    'brent'      : ['brent', 'crude', 'oil', 'petroleum', 'barrel', 'opec'],
    'wti'        : ['wti', 'west texas', 'crude', 'oil', 'petroleum', 'barrel'],
    'natural_gas': ['natural gas', 'lng', 'gas', 'methane', 'henry hub'],
    'silver'     : ['silver', 'precious metal', 'xag', 'spot silver', 'comex silver'],
}

def keyword_topic_sim(text: str, keywords: list) -> float:
    text_lower = text.lower()
    hits = sum(1 for kw in keywords if kw in text_lower)
    return min(hits / max(len(keywords), 1), 1.0)

EVENT_KEYWORDS = {
    'supply': [
        'supply', 'production', 'output cut', 'output increase', 'OPEC', 'quota',
        'upstream', 'drilling', 'rig count', 'shale', 'tight oil', 'offshore',
        'oil field', 'gas field', 'mine output', 'smelter output', 'crude production',
        'pumping', 'well', 'extraction', 'reserves', 'producer nation',
        'LNG export', 'gas export', 'silver mining', 'metal output',
    ],
    'demand': [
        'demand', 'consumption', 'import', 'buyer', 'purchasing',
        'GDP growth', 'economic growth', 'industrial activity', 'manufacturing activity',
        'fuel demand', 'jet fuel', 'gasoline demand', 'energy demand',
        'China demand', 'India demand', 'Asia demand', 'emerging market demand',
        'construction demand', 'infrastructure spending', 'EV demand',
        'industrial demand', 'utility demand',
    ],
    'inventory': [
        'inventory', 'stockpile', 'storage level', 'crude stock',
        'EIA report', 'API report', 'crude draw', 'crude build',
        'drawdown', 'stock change', 'warehouse stock', 'depot',
        'strategic petroleum reserve', 'SPR', 'LME stock', 'Comex inventory',
        'gas storage', 'tank level', 'days of supply',
    ],
    'transport': [
        'tanker', 'shipping', 'vessel', 'freight rate', 'cargo',
        'Strait of Hormuz', 'Suez Canal', 'Red Sea', 'pipeline',
        'export terminal', 'port blockage', 'logistics',
        'transport disruption', 'route closure', 'chokepoint',
        'loading terminal', 'oil route',
    ],
    'policy': [
        'OPEC meeting', 'OPEC decision', 'sanction', 'tariff', 'embargo',
        'export ban', 'import duty', 'government policy', 'energy ministry',
        'IEA', 'carbon tax', 'climate policy', 'energy regulation',
        'trade deal', 'trade agreement', 'geopolit', 'diplomatic',
        'subsidy cut', 'price cap', 'windfall tax',
    ],
    'weather': [
        'hurricane', 'tropical storm', 'cyclone', 'typhoon', 'tornado',
        'flood', 'drought', 'weather disruption', 'natural disaster',
        'force majeure', 'cold snap', 'heatwave', 'winter storm',
        'monsoon', 'freeze', 'blizzard', 'extreme weather',
    ],
    'macro': [
        'US dollar', 'USD index', 'DXY', 'interest rate', 'Fed rate',
        'Federal Reserve', 'inflation data', 'CPI', 'recession risk',
        'economic slowdown', 'GDP miss', 'trade war', 'currency move',
        'central bank', 'monetary policy', 'yield curve', 'bond yield',
        'risk appetite', 'risk-off', 'market sell-off',
    ],
    'outage': [
        'outage', 'unplanned shutdown', 'maintenance shutdown', 'downtime',
        'offline', 'refinery fire', 'plant explosion', 'facility incident',
        'capacity cut', 'force majeure', 'technical failure',
        'pipeline leak', 'mine closure', 'smelter closure', 'mill closure',
        'production halt', 'disruption at',
    ],
}

EVENT_CATEGORIES = list(EVENT_KEYWORDS.keys())

def classify_events(text: str) -> dict:
    text_lower = text.lower()
    return {
        cat: int(any(kw.lower() in text_lower for kw in kws))
        for cat, kws in EVENT_KEYWORDS.items()
    }

# ── Caches ────────────────────────────────────────────────────────────────────
TOPIC_CACHE = OUT_DIR / f'topic_scores_{TARGET_COMMODITY}.csv'
EVENT_CACHE = OUT_DIR / f'event_labels_{TARGET_COMMODITY}.csv'

# Auto-invalidate caches that used old 'date' column
for _cache in [TOPIC_CACHE, EVENT_CACHE]:
    if _cache.exists():
        if 'trading_day' not in pd.read_csv(_cache, nrows=0).columns.tolist():
            print(f'Old cache ({_cache.name}) — deleting...')
            _cache.unlink()

# Stage 1: keyword relevance (< 1 second)
if TOPIC_CACHE.exists():
    print('Loading cached topic relevance scores...')
    topic_df = pd.read_csv(TOPIC_CACHE, parse_dates=['trading_day'])
else:
    print('Computing keyword-based topic relevance...')
    kws = TOPIC_MAP[TARGET_COMMODITY]
    topic_df = news_df[['trading_day']].copy()
    topic_df['topic_sim'] = news_df['text'].apply(lambda t: keyword_topic_sim(t, kws))
    topic_df.to_csv(TOPIC_CACHE, index=False)
    print(f'Saved → {TOPIC_CACHE.name}')

# Stage 2: event classification (< 2 seconds)
if EVENT_CACHE.exists():
    print('Loading cached event labels...')
    event_df = pd.read_csv(EVENT_CACHE, parse_dates=['trading_day'])
else:
    print('Classifying event categories...')
    labels   = news_df['text'].apply(classify_events).apply(pd.Series)
    event_df = pd.concat([news_df[['trading_day']], labels], axis=1)
    event_df.to_csv(EVENT_CACHE, index=False)
    print(f'Saved → {EVENT_CACHE.name}')

# ── Direct column assignment (NOT merge — avoids many-to-many RAM explosion) ──
if 'topic_sim' not in sent_df.columns:
    sent_df['topic_sim'] = topic_df['topic_sim'].values

for cat in EVENT_CATEGORIES:
    if cat not in sent_df.columns:
        sent_df[cat] = event_df[cat].values

print(f'\nEvent category coverage:')
for cat in EVENT_CATEGORIES:
    print(f'  {cat:<12s}: {sent_df[cat].mean()*100:.1f}%')
unclassified = (sent_df[EVENT_CATEGORIES].sum(axis=1) == 0).mean() * 100
print(f'  {"(none)":12s}: {unclassified:.1f}%')
print(f'\ntopic_sim  min={topic_df["topic_sim"].min():.3f}  '
      f'mean={topic_df["topic_sim"].mean():.3f}  '
      f'max={topic_df["topic_sim"].max():.3f}')

In [ ]:
# ── 6. Daily Sentiment Aggregation (commodity-event aware) ───────────────────
#
# POLARITY MAP: FinBERT sign ≠ price-impact sign for supply-side categories.
# price_signal = sentiment × polarity
#   +1  demand/macro  →  positive sentiment = bullish price
#   -1  supply/inventory/transport/policy/weather/outage
#       →  negative sentiment (disruption) = bullish price

CATEGORY_POLARITY = {
    'supply'    : -1, 'demand'    : +1, 'inventory' : -1, 'transport' : -1,
    'policy'    : -1, 'weather'   : -1, 'macro'     : +1, 'outage'    : -1,
}

def aggregate_daily(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df['tw_sentiment'] = df['sentiment_score'] * df['topic_sim']

    overall = df.groupby('trading_day').agg(
        avg_sentiment     = ('sentiment_score', 'mean'),
        sentiment_std     = ('sentiment_score', 'std'),
        news_volume       = ('sentiment_score', 'count'),
        topic_w_sentiment = ('tw_sentiment',    'mean'),
    ).reset_index()
    overall['sentiment_std']  = overall['sentiment_std'].fillna(0)
    overall['log_news_vol']   = np.log1p(overall['news_volume'])
    overall['sentiment_volw'] = overall['avg_sentiment'] * overall['log_news_vol']

    cat_frames = []
    for cat in EVENT_CATEGORIES:
        cat_arts = df[df[cat] == 1]
        if cat_arts.empty:
            stub = overall[['trading_day']].copy()
            stub[f'{cat}_sentiment']    = 0.0
            stub[f'{cat}_price_signal'] = 0.0
            stub[f'{cat}_count']        = 0
            cat_frames.append(stub)
            continue
        agg = cat_arts.groupby('trading_day').agg(
            **{f'{cat}_sentiment': ('sentiment_score', 'mean'),
               f'{cat}_count'    : ('sentiment_score', 'count')}
        ).reset_index()
        agg[f'{cat}_price_signal'] = agg[f'{cat}_sentiment'] * CATEGORY_POLARITY[cat]
        cat_frames.append(agg)

    daily = overall.copy()
    for frame in cat_frames:
        daily = daily.merge(frame, on='trading_day', how='left')

    for cat in EVENT_CATEGORIES:
        daily[f'{cat}_sentiment']    = daily[f'{cat}_sentiment'].fillna(0)
        daily[f'{cat}_price_signal'] = daily[f'{cat}_price_signal'].fillna(0)
        daily[f'{cat}_count']        = daily[f'{cat}_count'].fillna(0).astype(int)

    signal_cols = [f'{cat}_price_signal' for cat in EVENT_CATEGORIES]
    daily['composite_price_signal'] = daily[signal_cols].sum(axis=1)
    return daily

daily_df = aggregate_daily(sent_df)
print(f'Daily records: {len(daily_df)}')
print(f'Columns      : {list(daily_df.columns)}')

In [ ]:
# ── 7. Merge & Feature Engineering ───────────────────────────────────────────
def build_features(price_df: pd.DataFrame, daily_df: pd.DataFrame) -> pd.DataFrame:
    df = price_df.copy()
    df['price'] = pd.to_numeric(df['price'], errors='coerce')   # safety guard

    # ── Price-derived features ─────────────────────────────────────────────
    df['log_return']  = np.log(df['price']).diff()
    df['roll5_ret']   = df['price'].pct_change().rolling(5).mean()
    df['roll10_ret']  = df['price'].pct_change().rolling(10).mean()
    df['roll20_vol']  = df['price'].pct_change().rolling(20).std()
    df['ma5_ratio']   = df['price'] / df['price'].rolling(5).mean()
    df['ma20_ratio']  = df['price'] / df['price'].rolling(20).mean()

    # ── Merge sentiment features on trading_day ────────────────────────────
    df = df.rename(columns={'date': 'trading_day'})
    df = df.merge(daily_df, on='trading_day', how='left')

    fill_cols = (
        ['avg_sentiment', 'sentiment_std', 'news_volume',
         'topic_w_sentiment', 'sentiment_volw', 'log_news_vol',
         'composite_price_signal']
        + [f'{cat}_sentiment'    for cat in EVENT_CATEGORIES]
        + [f'{cat}_price_signal' for cat in EVENT_CATEGORIES]
        + [f'{cat}_count'        for cat in EVENT_CATEGORIES]
    )
    for c in fill_cols:
        if c in df.columns:
            df[c] = df[c].fillna(0)

    # ── Target: next-day direction ─────────────────────────────────────────
    df['target_return'] = df['log_return'].shift(-1)
    df = df.dropna().reset_index(drop=True)
    return df

merged = build_features(price_df, daily_df)
print(f'Merged shape : {merged.shape}')
print(f'Date range   : {merged["trading_day"].min().date()} → {merged["trading_day"].max().date()}')
merged[['trading_day', 'price', 'log_return',
        'avg_sentiment', 'composite_price_signal',
        'supply_price_signal', 'demand_price_signal', 'target_return']].tail(5)

In [ ]:
# ── 8. Train / Val / Test Split + Scaling + Dataset ──────────────────────────
FEATURE_COLS = [
    # Price-derived (6)
    'log_return', 'roll5_ret', 'roll10_ret', 'roll20_vol', 'ma5_ratio', 'ma20_ratio',
    # Overall sentiment (3)
    'avg_sentiment', 'sentiment_std', 'log_news_vol',
    # Composite directional signal (1)
    'composite_price_signal',
    # Per-category price signals (8)
    'supply_price_signal', 'demand_price_signal', 'inventory_price_signal',
    'transport_price_signal', 'policy_price_signal', 'weather_price_signal',
    'macro_price_signal', 'outage_price_signal',
    # Per-category article counts (8)
    'supply_count', 'demand_count', 'inventory_count', 'transport_count',
    'policy_count', 'weather_count', 'macro_count', 'outage_count',
]

missing = [c for c in FEATURE_COLS if c not in merged.columns]
assert not missing, f'Missing columns: {missing}'
print(f'Feature count : {len(FEATURE_COLS)}')

n         = len(merged)
train_end = int(n * TRAIN_RATIO)
val_end   = int(n * (TRAIN_RATIO + VAL_RATIO))

train_df = merged.iloc[:train_end].copy()
val_df   = merged.iloc[train_end:val_end].copy()
test_df  = merged.iloc[val_end:].copy()
print(f'Split  →  Train: {len(train_df)}  Val: {len(val_df)}  Test: {len(test_df)}')

scaler_X = StandardScaler()
X_train  = scaler_X.fit_transform(train_df[FEATURE_COLS].values)
X_val    = scaler_X.transform(val_df[FEATURE_COLS].values)
X_test   = scaler_X.transform(test_df[FEATURE_COLS].values)

y_train_bin  = (train_df['target_return'].values > 0).astype(np.float32).reshape(-1, 1)
y_val_bin    = (val_df['target_return'].values   > 0).astype(np.float32).reshape(-1, 1)
y_test_bin   = (test_df['target_return'].values  > 0).astype(np.float32).reshape(-1, 1)
y_train_flat = y_train_bin.flatten().astype(int)
y_val_flat   = y_val_bin.flatten().astype(int)
y_test_flat  = y_test_bin.flatten().astype(int)

up_pct = y_train_flat.mean() * 100
print(f'Class balance (train): {up_pct:.1f}% up / {100-up_pct:.1f}% down')

class SequenceDataset(Dataset):
    def __init__(self, X, y, seq_len):
        self.X, self.y, self.seq_len = torch.FloatTensor(X), torch.FloatTensor(y), seq_len
    def __len__(self):
        return len(self.X) - self.seq_len
    def __getitem__(self, idx):
        return self.X[idx : idx + self.seq_len], self.y[idx + self.seq_len]

train_ds = SequenceDataset(X_train, y_train_bin, SEQUENCE_LEN)
val_ds   = SequenceDataset(X_val,   y_val_bin,   SEQUENCE_LEN)
test_ds  = SequenceDataset(X_test,  y_test_bin,  SEQUENCE_LEN)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False)

print(f'Sequences  →  Train: {len(train_ds)}  Val: {len(val_ds)}  Test: {len(test_ds)}')

In [ ]:
# ── 8b. Baselines: LR / RF / GBM / XGBoost / LightGBM ───────────────────────
#
# Flat features (no sequence window). XGBoost + LightGBM use early stopping
# on the val set. sklearn models fit on train+val combined.
# Missing packages are skipped gracefully.

from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

try:
    import xgboost as xgb
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print('xgboost not installed  →  pip install xgboost')

try:
    import lightgbm as lgb
    HAS_LGB = True
except ImportError:
    HAS_LGB = False
    print('lightgbm not installed  →  pip install lightgbm')

X_trval = np.vstack([X_train, X_val])
y_trval = np.concatenate([y_train_flat, y_val_flat])

BASELINES = {
    'LogisticRegression': LogisticRegression(max_iter=1000, C=0.1),
    'RandomForest': RandomForestClassifier(
        n_estimators=200, max_depth=4, min_samples_leaf=10, random_state=42),
    'GradientBoosting': GradientBoostingClassifier(
        n_estimators=100, max_depth=3, learning_rate=0.05,
        min_samples_leaf=10, random_state=42),
}
if HAS_XGB:
    BASELINES['XGBoost'] = xgb.XGBClassifier(
        n_estimators=500, max_depth=3, learning_rate=0.03,
        subsample=0.8, colsample_bytree=0.8, min_child_weight=5,
        reg_alpha=0.1, reg_lambda=1.0, eval_metric='logloss',
        early_stopping_rounds=30, random_state=42, verbosity=0)
if HAS_LGB:
    BASELINES['LightGBM'] = lgb.LGBMClassifier(
        n_estimators=500, max_depth=4, learning_rate=0.03, num_leaves=15,
        min_child_samples=10, subsample=0.8, colsample_bytree=0.8,
        reg_alpha=0.1, reg_lambda=1.0, random_state=42, verbose=-1)

print(f'{"Model":<22} {"Acc":>6} {"F1":>6} {"AUC":>6}')
print('-' * 44)

baseline_results = {}
for name, clf in BASELINES.items():
    if name == 'XGBoost' and HAS_XGB:
        clf.fit(X_train, y_train_flat,
                eval_set=[(X_val, y_val_flat)], verbose=False)
    elif name == 'LightGBM' and HAS_LGB:
        clf.fit(X_train, y_train_flat,
                eval_set=[(X_val, y_val_flat)],
                callbacks=[lgb.early_stopping(30, verbose=False),
                            lgb.log_evaluation(-1)])
    else:
        clf.fit(X_trval, y_trval)

    preds = clf.predict(X_test)
    probs = clf.predict_proba(X_test)[:, 1]
    acc   = accuracy_score(y_test_flat, preds) * 100
    f1    = f1_score(y_test_flat, preds, zero_division=0)
    auc   = roc_auc_score(y_test_flat, probs)
    baseline_results[name] = dict(acc=acc, f1=f1, auc=auc, preds=preds, probs=probs)
    print(f'{name:<22} {acc:>5.1f}%  {f1:>5.3f}  {auc:>5.3f}')

naive_acc = max(y_test_flat.mean(), 1 - y_test_flat.mean()) * 100
print(f'\nNaive baseline (majority class): {naive_acc:.1f}%')

# Feature importances from best tree model
tree_order = ['LightGBM', 'XGBoost', 'GradientBoosting', 'RandomForest']
tree_name  = next((t for t in tree_order if t in BASELINES), None)
if tree_name:
    importances = pd.Series(
        BASELINES[tree_name].feature_importances_, index=FEATURE_COLS
    ).sort_values(ascending=False)
    print(f'\nTop-10 feature importances ({tree_name}):')
    for feat, imp in importances.head(10).items():
        print(f'  {feat:<28s} {imp:.4f}  {"█" * int(imp*200)}')
    importances.to_csv(
        OUT_DIR / f'feature_importances_{TARGET_COMMODITY}.csv', header=['importance'])

In [ ]:
# ── 9. Binary LSTM Model ─────────────────────────────────────────────────────
# 1 layer, 32 hidden, ~8k params — appropriate for ~300 training sequences.
# Output: single logit (BCEWithLogitsLoss applies sigmoid internally).

class BinaryLSTM(nn.Module):
    def __init__(self, input_size, hidden_size=32, dropout=0.4):
        super().__init__()
        self.lstm = nn.LSTM(input_size=input_size, hidden_size=hidden_size,
                            num_layers=1, batch_first=True)
        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_size, hidden_size // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size // 2, 1),
        )
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.head(out[:, -1])

HIDDEN_SIZE = 32
DROPOUT     = 0.4
model = BinaryLSTM(input_size=len(FEATURE_COLS),
                   hidden_size=HIDDEN_SIZE, dropout=DROPOUT).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model      : BinaryLSTM (1 layer, {HIDDEN_SIZE} hidden)')
print(f'Input      : {len(FEATURE_COLS)} features × {SEQUENCE_LEN} timesteps')
print(f'Parameters : {n_params:,}')
print(model)

In [ ]:
# ── 10. Training Loop ────────────────────────────────────────────────────────
criterion  = nn.BCEWithLogitsLoss()
optimizer  = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler  = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', patience=8, factor=0.5)

MODEL_PATH       = OUT_DIR / f'lstm_binary_{TARGET_COMMODITY}_best.pt'
train_losses, val_losses = [], []
train_accs,   val_accs   = [], []
best_val_loss    = float('inf')
patience_counter = 0

print(f'Training {EPOCHS} epochs on {DEVICE} ...\n')

for epoch in range(1, EPOCHS + 1):
    model.train()
    t_loss, t_correct, t_total = 0.0, 0, 0
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        logits = model(xb)
        loss   = criterion(logits, yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        t_loss    += loss.item()
        preds      = (torch.sigmoid(logits) >= 0.5).float()
        t_correct += (preds == yb).sum().item()
        t_total   += yb.numel()
    t_loss /= len(train_loader)
    t_acc   = t_correct / t_total * 100

    model.eval()
    v_loss, v_correct, v_total = 0.0, 0, 0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            logits  = model(xb)
            v_loss += criterion(logits, yb).item()
            preds   = (torch.sigmoid(logits) >= 0.5).float()
            v_correct += (preds == yb).sum().item()
            v_total   += yb.numel()
    v_loss /= len(val_loader)
    v_acc   = v_correct / v_total * 100

    train_losses.append(t_loss);  val_losses.append(v_loss)
    train_accs.append(t_acc);     val_accs.append(v_acc)
    scheduler.step(v_loss)

    if v_loss < best_val_loss:
        best_val_loss    = v_loss
        patience_counter = 0
        torch.save(model.state_dict(), MODEL_PATH)
    else:
        patience_counter += 1

    if epoch % 10 == 0 or epoch == 1:
        lr_now = optimizer.param_groups[0]['lr']
        print(f'Epoch {epoch:3d}/{EPOCHS}  '
              f'loss={t_loss:.4f}/{v_loss:.4f}  '
              f'acc={t_acc:.1f}%/{v_acc:.1f}%  '
              f'lr={lr_now:.1e}  pat={patience_counter}')

    if patience_counter >= EARLY_STOP_PAT:
        print(f'\nEarly stop at epoch {epoch}.')
        break

print(f'\nBest val loss: {best_val_loss:.4f}  →  {MODEL_PATH.name}')

In [ ]:
# ── 11. Evaluation ───────────────────────────────────────────────────────────
from sklearn.metrics import confusion_matrix, roc_auc_score, f1_score, accuracy_score

model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE, weights_only=True))
model.eval()

all_logits, all_labels = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        all_logits.append(model(xb.to(DEVICE)).cpu())
        all_labels.append(yb)

logits_np = torch.cat(all_logits).numpy().flatten()
labels_np = torch.cat(all_labels).numpy().flatten().astype(int)
probs_lstm = torch.sigmoid(torch.tensor(logits_np)).numpy()
preds_lstm = (probs_lstm >= 0.5).astype(int)

# Baselines predict on all X_test rows; LSTM starts at SEQUENCE_LEN offset.
baseline_offset = len(y_test_flat) - len(labels_np)

def report(name, preds, probs, labels):
    acc = accuracy_score(labels, preds) * 100
    f1  = f1_score(labels, preds, zero_division=0)
    auc = roc_auc_score(labels, probs)
    cm  = confusion_matrix(labels, preds)
    return dict(name=name, acc=acc, f1=f1, auc=auc, cm=cm, preds=preds, probs=probs)

naive_preds = np.full_like(labels_np, int(y_train_flat.mean() >= 0.5))
naive_probs = np.full(len(labels_np), y_train_flat.mean())

results = [report('Naive (majority)', naive_preds, naive_probs, labels_np)]
for name in ['LogisticRegression', 'RandomForest', 'GradientBoosting',
             'XGBoost', 'LightGBM']:
    if name in baseline_results:
        r = baseline_results[name]
        results.append(report(
            name,
            r['preds'][baseline_offset:],
            r['probs'][baseline_offset:],
            labels_np,
        ))
results.append(report('BinaryLSTM', preds_lstm, probs_lstm, labels_np))

print('=' * 62)
print(f'  TEST SET  ({TARGET_COMMODITY.upper()}, {len(labels_np)} samples)')
print('=' * 62)
print(f'  {"Model":<22} {"Acc":>6}  {"F1":>6}  {"AUC":>6}')
print('  ' + '-' * 50)
best_acc = max(r['acc'] for r in results if r['name'] != 'Naive (majority)')
for r in results:
    marker = ' ◀ best' if r['acc'] == best_acc else ''
    print(f'  {r["name"]:<22} {r["acc"]:>5.1f}%  {r["f1"]:>6.3f}  {r["auc"]:>6.3f}{marker}')
print('=' * 62)

lstm_r = next(r for r in results if r['name'] == 'BinaryLSTM')
cm = lstm_r['cm']
print(f'\nConfusion matrix (BinaryLSTM):')
if cm.shape == (2, 2):
    print(f'  Pred →    Down   Up')
    print(f'  Act Down [{cm[0,0]:>4}  {cm[0,1]:>4}]')
    print(f'  Act Up   [{cm[1,0]:>4}  {cm[1,1]:>4}]')

with open(OUT_DIR / f'metrics_binary_{TARGET_COMMODITY}.json', 'w') as f:
    json.dump([{k: v for k, v in r.items() if k not in ('cm','preds','probs')}
               for r in results], f, indent=2)

In [ ]:
# ── 12. Visualisation ────────────────────────────────────────────────────────
from sklearn.metrics import roc_curve

COLORS = {
    'Naive (majority)'  : 'grey',
    'LogisticRegression': 'steelblue',
    'RandomForest'      : 'darkorange',
    'GradientBoosting'  : 'green',
    'XGBoost'           : 'purple',
    'LightGBM'          : 'teal',
    'BinaryLSTM'        : 'crimson',
}

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle(
    f'Binary Classification Results — {TARGET_COMMODITY.upper()}\n'
    f'Test set: {len(labels_np)} samples  |  Sequence len: {SEQUENCE_LEN}',
    fontsize=13, fontweight='bold')

# (a) Training curves
ax = axes[0, 0]
ax.plot(train_losses, label='Train BCE', lw=1.5)
ax.plot(val_losses,   label='Val BCE',   lw=1.5)
ax2 = ax.twinx()
ax2.plot(train_accs, '--', lw=1, alpha=0.7, color='tab:green', label='Train Acc')
ax2.plot(val_accs,   '--', lw=1, alpha=0.7, color='tab:red',   label='Val Acc')
ax2.set_ylabel('Accuracy %', color='grey')
ax.set_title('Training Curves'); ax.set_xlabel('Epoch'); ax.set_ylabel('BCE Loss')
lines1, labs1 = ax.get_legend_handles_labels()
lines2, labs2 = ax2.get_legend_handles_labels()
ax.legend(lines1+lines2, labs1+labs2, fontsize=8); ax.grid(True, alpha=0.3)

# (b) ROC curves
ax = axes[0, 1]
for r in results:
    fpr, tpr, _ = roc_curve(labels_np, r['probs'])
    lw = 2.5 if r['name'] == 'BinaryLSTM' else 1.2
    color = COLORS.get(r['name'], 'black')
    ax.plot(fpr, tpr, label=f'{r["name"]} ({r["auc"]:.3f})', color=color, lw=lw)
ax.plot([0,1],[0,1],'k--',lw=0.8,label='Random')
ax.set_title('ROC Curves'); ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
ax.legend(fontsize=7); ax.grid(True, alpha=0.3)

# (c) Accuracy bar
ax = axes[1, 0]
names = [r['name'] for r in results]
accs  = [r['acc']  for r in results]
bars  = ax.bar(range(len(names)), accs,
               color=[COLORS.get(n,'black') for n in names], alpha=0.85, edgecolor='white')
ax.axhline(50, color='black', lw=1, linestyle='--', label='Random (50%)')
for bar, val in zip(bars, accs):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
            f'{val:.1f}%', ha='center', fontsize=8, fontweight='bold')
ax.set_xticks(range(len(names)))
ax.set_xticklabels(names, rotation=15, ha='right', fontsize=8)
ax.set_title('Directional Accuracy'); ax.set_ylabel('Accuracy %')
ax.set_ylim(35, max(accs)+10); ax.legend(); ax.grid(True, alpha=0.3, axis='y')

# (d) Feature importances
ax = axes[1, 1]
if 'importances' in dir():
    top10 = importances.head(10)
    ax.barh(range(len(top10)), top10.values[::-1], color='steelblue', alpha=0.8)
    ax.set_yticks(range(len(top10)))
    ax.set_yticklabels(top10.index[::-1], fontsize=9)
    ax.set_title(f'Top-10 Feature Importances ({tree_name})')
    ax.set_xlabel('Importance'); ax.grid(True, alpha=0.3, axis='x')
else:
    ax.text(0.5, 0.5, 'No tree model importances available',
            ha='center', va='center', transform=ax.transAxes)

plt.tight_layout()
plot_path = OUT_DIR / f'results_v3_{TARGET_COMMODITY}.png'
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Plot saved → {plot_path}')